# 01 -- Theory walkthrough (Props 1-2)

Narrative only, per `project.md` SS9: this notebook imports from `src/qapinn` and reads
committed results; it does not re-derive anything, and none of it is a GPU workload.

Verifies, numerically, on real PDE instances:
- **Proposition 1**: a data-re-uploading circuit's realized frequency spectrum is
  exactly the set SMCD's design card claims (`ReuploadCircuit.frequencies()` matches
  `DesignCard.omega_set`).
- **Proposition 2**: SMCD's target spectrum computation (Fourier symbol of the
  operator, forcing spectrum).

In [ ]:
import numpy as np
from qapinn.pdes.poisson import Poisson
from qapinn.smcd.design import smcd
from qapinn.smcd.symbol import analytic_spectrum
from qapinn.models.circuits import ReuploadCircuit


def _as_rows(a):
    # (n,) -> (n,1); (n,d) stays (n,d) -- circuit.frequencies() returns a flat 1-D
    # array for a 1-D PDE but a [n, d] array for a d-dimensional one (e.g. Helmholtz).
    a = np.asarray(a, dtype=float)
    return a.reshape(-1, 1) if a.ndim == 1 else a


def freq_sets_match(realized, omega_set, decimals=6):
    # Set-equality check between the circuit's realized frequencies and the design
    # card's own omega_set, independent of row order.
    r, t = _as_rows(realized), _as_rows(omega_set)
    if r.shape != t.shape:
        return False
    r_rows = {tuple(np.round(row, decimals)) for row in r}
    t_rows = {tuple(np.round(row, decimals)) for row in t}
    return r_rows == t_rows

## Proposition 2: analytic target spectrum for Poisson

In [ ]:
pde = Poisson(alpha=0.3)
target = analytic_spectrum(pde, eps=1e-3)
print(f"target spectrum: {target.omega.shape[0]} candidate frequencies, "
      f"{int(target.support.sum())} above the eps={1e-3} support threshold")
print("first 5 supported omega:", target.omega[target.support][:5].ravel())

## Proposition 1: circuit's realized frequencies == SMCD's design card

In [ ]:
card = smcd(pde)
print(f"SMCD design: n_qubits={card.n_qubits}, n_layers={card.n_layers}, "
      f"entangler={card.entangler!r}, coverage={card.coverage:.4f}, "
      f"predicted_benefit={card.predicted_benefit:.4f}")

circuit = ReuploadCircuit(
    n_qubits=card.n_qubits, n_layers=card.n_layers, scalings=card.scalings,
    wire_to_dim=card.wire_to_dim, entangler=card.entangler, observable=card.observable,
)
realized = circuit.frequencies()
match = freq_sets_match(realized, card.omega_set)
print(f"realized frequency count: {len(_as_rows(realized))}")
print(f"realized == design card omega_set (Prop. 1 holds): {match}")
assert match, "Proposition 1 violated -- this should never happen for a real qapinn build"

## Same check, a second PDE instance (Helmholtz, k=10)

Prop. 1/2 are claimed to hold for every PDE this project supports, not just Poisson --
repeat on a structurally different instance (2-D, oscillatory forcing, not diffusive).

In [ ]:
from qapinn.pdes.helmholtz import Helmholtz

pde2 = Helmholtz(k=10.0, a1=3.0, a2=1.0)
card2 = smcd(pde2)
circuit2 = ReuploadCircuit(
    n_qubits=card2.n_qubits, n_layers=card2.n_layers, scalings=card2.scalings,
    wire_to_dim=card2.wire_to_dim, entangler=card2.entangler, observable=card2.observable,
)
realized2 = circuit2.frequencies()
match2 = freq_sets_match(realized2, card2.omega_set)
print(f"Helmholtz(k=10): n_qubits={card2.n_qubits}, n_layers={card2.n_layers}, "
      f"coverage={card2.coverage:.4f}")
print(f"realized == design card omega_set (Prop. 1 holds): {match2}")
assert match2